## Резюме

В этой лекции мы рассмотрели:
1. Тест на единичный корень и более глубокое понимание теста ADF.
2. Возврат к среднему и как его проверить с помощью ADF.
3. Показатель Херста: как его вычислить и его применение в финансах.
4. Геометрическое броуновское движение: определение и применение для моделирования акций.

## Упражнение

- Загрузите и нанесите на график исторические цены закрытия Tesla (``TSLA``) и Equinor (``EQNR``) за годы ``2019-12-31``- ``2022-12-31``.
- Для каждого временного ряда:
    - Проверьте, выглядит ли временной ряд стационарным.
    - Вычислите коэффициент Херста для обоих временных рядов.
    - В какие акции вы хотели бы инвестировать? Обоснуйте свой ответ на основе тестов и значения $H$.
    - Смоделируйте цены акций с помощью GBM.

- Какая симуляция кажется более надежной? Для Tesla или Equinor?
- Чтобы обосновать свой ответ:
1. вычислите симуляцию не менее 100 раз.
2. Вычислите MSE между истинными ценами акций и смоделированными.
3. Сравните ожидаемое значение MAPE для двух акций.

In [3]:
import yfinance as yf
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller

stocks = ['TSLA', 'EQNR']
data = yf.download(stocks, start='2019-12-31', end='2022-12-31')['Close']

# проверка на стационарность
for s in stocks:
    res = adfuller(data[s])
    print(f'ряд {s} p-value: {res[1]}')

# херст для понимания тренда
def get_hurst(ts):
    lags = range(2, 100)
    tau = [np.sqrt(np.std(np.subtract(ts[lag:], ts[:-lag]))) for lag in lags]
    poly = np.polyfit(np.log(lags), np.log(tau), 1)
    return poly[0] * 2.0

h_tsla = get_hurst(data['TSLA'].values)
h_eqnr = get_hurst(data['EQNR'].values)

def gbm_sim(s0, mu, sigma, n_steps, n_sims=100):
    dt = 1/252
    st = np.zeros((n_sims, n_steps))
    st[:, 0] = s0
    for i in range(n_sims):
        for j in range(1, n_steps):
            st[i, j] = st[i, j-1] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * np.random.standard_normal())
    return st

# считаем параметры доходности
rets1 = data['TSLA'].pct_change().dropna()
mu1, sigma1 = rets1.mean() * 252, rets1.std() * np.sqrt(252)

sim_tsla = gbm_sim(data['TSLA'].iloc[0], mu1, sigma1, len(data))

rets2 = data['EQNR'].pct_change().dropna()
mu2, sigma2 = rets2.mean() * 252, rets2.std() * np.sqrt(252)
sim_eqnr = gbm_sim(data['EQNR'].iloc[0], mu2, sigma2, len(data))

# считаем ошибки для обоснования
mse_tsla = np.mean([np.mean((data['TSLA'].values - sim_tsla[i])**2) for i in range(100)])
mse_eqnr = np.mean([np.mean((data['EQNR'].values - sim_eqnr[i])**2) for i in range(100)])
mape_tsla = np.mean([np.mean(np.abs((data['TSLA'].values - sim_tsla[i]) / data['TSLA'].values)) for i in range(100)])
mape_eqnr = np.mean([np.mean(np.abs((data['EQNR'].values - sim_eqnr[i]) / data['EQNR'].values)) for i in range(100)])

print(f'tsla h: {h_tsla} mse: {mse_tsla} mape: {mape_tsla}')
print(f'eqnr h: {h_eqnr} mse: {mse_eqnr} mape: {mape_eqnr}')

[*********************100%***********************]  2 of 2 completed


ряд TSLA p-value: 0.3492855391367856
ряд EQNR p-value: 0.9028741234149483
tsla h: 0.46262991516271335 mse: 64758.51028686825 mape: 0.706002329628179
eqnr h: 0.33155079027290935 mse: 303.7169260123365 mape: 0.6529310017492465
